# Talking to ROS 2 from a browser — the simple way

This notebook shows the **simplest possible** way a browser can exchange data with a zenoh-routed ROS 2 graph: a plain HTTP `PUT`/`GET` against the [zenohd REST plugin](https://zenoh.io/docs/manual/plugin-rest/), using nothing but the standard-library-adjacent `pyodide.http` module. It is *not* a WASM build of ROS 2 — it is a lightweight complement to the full, genuinely-compiled `rclpy` + `numpy` WASM32 demo that talks to ROS 2 over `rmw_zenoh_pico`, which lives at [Tobias-Fischer/ros2-emscripten-zenoh-demo](https://github.com/Tobias-Fischer/ros2-emscripten-zenoh-demo).

If you don't have a zenoh router running locally, the cells below will tell you how to start one.

In [ ]:
# Edit this if your zenoh router's REST plugin is running somewhere else.
ZENOH_REST_URL = "http://127.0.0.1:8000/demo/from_jupyterlite"

CONNECT_HELP = (
    "Could not reach a zenoh REST endpoint at {url}.\n\n"
    "Most visitors won't have one running locally — to try this for real, start a zenoh "
    "router with its REST plugin enabled:\n\n"
    "    zenohd --rest-http-port 8000\n\n"
    "If you also want the full compiled-WASM rclpy demo running at the same time, add a "
    "websocket listener for it too:\n\n"
    "    zenohd --rest-http-port 8000 -l ws/127.0.0.1:7447\n"
)

In [ ]:
# PUT a value into the zenoh key/value store via the REST plugin.
import pyodide.http

try:
    response = await pyodide.http.pyfetch(
        ZENOH_REST_URL,
        method="PUT",
        body="hello from JupyterLite!",
    )
    print(f"PUT -> HTTP {response.status}")
except Exception as exc:
    print(CONNECT_HELP.format(url=ZENOH_REST_URL))
    print(f"(underlying error: {exc})")

In [ ]:
# GET it back out again to prove the round trip worked.
try:
    response = await pyodide.http.pyfetch(ZENOH_REST_URL, method="GET")
    body = await response.string()
    print(f"GET -> HTTP {response.status}")
    print(f"Value stored in zenoh: {body!r}")
except Exception as exc:
    print(CONNECT_HELP.format(url=ZENOH_REST_URL))
    print(f"(underlying error: {exc})")